# Parallel Papermill Runner

This notebook demonstrates how the class executes multiple parameter sets in parallel, how the submission loop can be stopped, and how execution can later resume automatically via `log.json`.

- each run writes to its own directory (for example `example_runs/0000/`, `example_runs/0001/`)
- completed runs are automatically tracked in `example_runs/log.json`
- **Ctrl+C stops the submission loop** (already running notebooks still finish)
- `request_stop()` stops the loop, not the notebooks that were already started
- `resume=True` automatically detects `log.json` and continues remaining runs

## Imports and Setup

Import the required standard library modules and load the parallel runner class used in the following steps.

In [ ]:
from pathlib import Path
import threading
import time

from papermill_parallel_runner import ParallelPapermillRunner

## Parameter Grid and Postprocessing

Define the parameter combinations for all notebook runs and provide a postprocessing callback that reports completed jobs.

In [ ]:
parameter_variations = {
    "a": [3, 5, 6, 7],
    "b": [9, 5, 3, 2],
}

def postprocessing(result):
    print(f"finished: {result.output_notebook.name} | {result.duration_seconds:.2f}s")
    time.sleep(1.0)

## First Pass with Controlled Stop

Create a runner without resume mode, start a short timer, and request a graceful stop to simulate an interrupted submission loop.

In [ ]:
runner = ParallelPapermillRunner(
    notebook=Path("example.ipynb"),
    parameter_variations=parameter_variations,
    files=[Path("test.txt")],
    workers=2,
    resume=False,
    postprocessing=postprocessing,
)

stop_timer = threading.Timer(1 , runner.request_stop)
stop_timer.start()
first_pass = runner.run()
stop_timer.cancel()

## Resume Remaining Runs

Initialize a new runner with `resume=True` to continue incomplete runs based on the log file and print the resolved log path.

In [ ]:
resume_runner = ParallelPapermillRunner(
    notebook=Path("example.ipynb"),
    parameter_variations=parameter_variations,
    files=[Path("test.txt")],
    workers=2,
    resume=True,
    postprocessing=postprocessing,
)

second_pass = resume_runner.run()

print("log file:", resume_runner.log_file.resolve())

In [ ]:
import scrapbook as sb

collected_results = []

for notebook_path in sorted(resume_runner.work_root_dir.glob("*/example.ipynb")):
    notebook = sb.read_notebook(str(notebook_path))
    scraps = notebook.scraps.data_dict

    collected_results.append(
        {
            "run_dir": notebook_path.parent.name,
            **scraps,
        }
    )

collected_results